In [2]:
import pandas as pd
import numpy as np
import os

# Exploratory Analysis

In [ ]:
## Load the data with the target columns
# !cut -f 1,8,15,18,21,22,26,67,68,71 /home/camilababo/Documents/DNAquaIMG/BOLD_Public.29-Nov-2024/BOLD_Public.29-Nov-2024.tsv | gzip -c - > bold-selected.tsv.gz

In [31]:
# Load the data
path = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected.tsv"
bold = pd.read_csv(path, on_bad_lines="skip", sep="\t")

/tmp/ipykernel_7321/1221080447.py:3: DtypeWarning: Columns (1,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  bold = pd.read_csv(path, on_bad_lines="skip", sep="\t")


In [4]:
bold.head()

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
0,AANIC003-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
1,AANIC002-10,BOLD:AAO2553,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCAGGTATAATTGGAACT...,658.0,COI-5P
2,AANIC058-10,BOLD:AAB9307,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATCTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
3,AANIC061-10,BOLD:ACE8261,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P
4,AANIC062-10,BOLD:AAF6782,Arthropoda,Geometridae,Arhodia,Arhodia lasiocamparia,NaN,AACATTATATTTTATTTTTGGTATTTGAGCTGGTATAATTGGAACT...,658.0,COI-5P


In [4]:
bold.tail()

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
17862992,ZYGMO209-10,BOLD:AAJ5548,Arthropoda,Zygaenidae,Adscita,Adscita obscura,NaN,AACACTTTACTTTATTTTTGGTATTTGATCAGGAATAGTTGGAACA...,658.0,COI-5P
17862993,ZYGMO290-10,BOLD:AAO1273,Arthropoda,Zygaenidae,Rhagades,Rhagades predotae,NaN,----------------------------------------------...,608.0,COI-5P
17862994,ZYGMO294-10,BOLD:AAO0371,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris minna,NaN,AACACTTTATTTTATTTTTGGAATTTGATCAGGAATAATTGGTACA...,658.0,COI-5P
17862995,ZYGMO295-10,BOLD:AAN9389,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris rjabovi,NaN,AACACTTTATTTTATCTTTGGAATCTGATCAGGAATAATTGGAACA...,658.0,COI-5P
17862996,ZYGMO296-10,BOLD:AAN9389,Arthropoda,Zygaenidae,Zygaenoprocris,Zygaenoprocris rjabovi,NaN,AACACTTTATTTTATCTTTGGAATCTGATCAGGAATAATTGGAACA...,658.0,COI-5P


In [17]:
# Check the total number of entries
bold.shape

(17862997, 10)

#### Check entries on 'Identification Method'

In [15]:
non_null_id = bold[bold['identification_method'] == 'Wikipidia']
non_null_id

,processid,bin_uri,phylum,family,genus,species,identification_method,nuc,nuc_basecount,marker_code
654690,AZSDS020-18,NaN,Arthropoda,Cerambycidae,NaN,NaN,Wikipidia,NaN,NaN,NaN


In [18]:
# Check the number of entries with COI-5P
count = bold['marker_code'].str.contains('COI-5P', na=False).sum()
int(count)

14790363

In [19]:
# Check the average length of entrie's nucleotide sequence
average_length = bold['nuc'].astype(str).apply(len).mean()
int(average_length)

610

In [13]:
# Check values with unique entries for 'identification method'
unique_idm = bold['identification_method'].unique()
np.savetxt("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/idm.txt",
           unique_idm, fmt='%s')

In [ ]:
# Check values with unique entries for 'marker code'
unique_mc = bold['marker_code'].unique()
np.savetxt("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/mc.txt",
           unique_mc, fmt='%s')

# Pre-processing

#### Filter out entries whose 'marker_code' is not 'COI-5P'

In [3]:
bold = bold[bold['marker_code'] == 'COI-5P']

#### Remove all entries whose sequence is inferior to 500 nucleotides

In [4]:
bold = bold[bold['nuc'].notna() & (bold['nuc_basecount'] > 500.0)]

#### Create column indicating highest taxon level available

In [5]:
bold['highest_tax_level'] = bold[['species', 'genus', 'family', 'phylum']].bfill(axis=1).iloc[:, 0]

#### Filter out entries whose presence of degenerate nucleotides is superior to 1% of the total sequence length

In [6]:
bold['N_count'] = bold['nuc'].str.count('N')

In [7]:
bold['prct_degenerate'] = (bold['N_count'] / bold['nuc_basecount']) * 100

In [8]:
bold = bold[bold['prct_degenerate'] <= 1]

#### Save the data

In [9]:
bold.shape

(14109605, 13)

In [10]:
bold.to_csv("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-init-preprocess.tsv", sep="\t", index=False)

#### Prepare data for taxonomic harmonization against GBIF

In [11]:
# Select highest taxonomic level column to run Tax. Harmonizer
bold_highest_tax = bold[['highest_tax_level']]
bold_highest_tax.rename(columns={'highest_tax_level': 'taxa'}, inplace=True)
bold_highest_tax.to_csv("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-highest_tax.tsv", sep="\t", index=False)

In [ ]:
# Transform the column into a list
list = bold_highest_tax['highest_tax_level'].tolist()

# Process Harmonization

In [3]:
# Load the harmonized data
path = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-highest_tax_harmonized.tsv"
bold_harmonized_taxa = pd.read_csv(path, on_bad_lines="skip", sep="\t")

In [38]:
# Preview the data
bold_harmonized_taxa.head()

,taxa,scientificName,rank,kingdom,phylum,class,order,family,genus
0,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
1,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
2,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
3,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia
4,Arhodia lasiocamparia,"Arhodia lasiocamparia Guenée, 1857",SPECIES,Animalia,Arthropoda,Insecta,Lepidoptera,Geometridae,Arhodia


In [28]:
# Check the total number of entries that retrieved no hits '-'
no_hit = bold_harmonized_taxa[bold_harmonized_taxa['scientificName'] == '-']
no_hit_taxa = no_hit['taxa'].to_list()
no_harmonized_entry_percent = len(no_hit_taxa) / len(bold_harmonized_taxa) * 100
f"Number of entries present in the dataset that did not retrieve a harmonized entry: {round(no_harmonized_entry_percent, 1)}%"

'Number of entries present in the dataset that did not retrieve a harmonized entry: 2.3%'

In [29]:
no_hit_taxa_unique = no_hit['taxa'].unique()
f"Number of unique taxa present in the dataset that did not retrieve a harmonized entry: {len(no_hit_taxa_unique)}"

'Number of unique taxa present in the dataset that did not retrieve a harmonized entry: 19458'

In [45]:
# # Write the unique taxa to a file for external checking
# with open("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/no_gbif_hit_taxa.txt", "w") as file:
#     for entry in no_hit_taxa_unique:
#         file.write(entry + "\n")

### Merge pre-processed BOLD with harmonized entries

In [4]:
# Load the initial pre-process data
path = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-selected-init-preprocess.tsv"
bold_init_process = pd.read_csv(path, on_bad_lines="skip", sep="\t")

/tmp/ipykernel_4479/1678772608.py:3: DtypeWarning: Columns (1,6) have mixed types. Specify dtype option on import or set low_memory=False.
  bold_init_process = pd.read_csv(path, on_bad_lines="skip", sep="\t")


In [5]:
new_cols = ['processid', 'bin_uri', 'identification_method', 'marker_code', 'nuc', 'nuc_basecount', 'highest_tax_level']
bold_init_process = bold_init_process[new_cols]

In [6]:
# Merge bold_init_process and bold_harmonized_taxa
merged_df = bold_init_process.join(bold_harmonized_taxa)


In [7]:
final_cols = ['processid', 'bin_uri', 'scientificName', 'rank', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'identification_method', 'marker_code', 'nuc', 'nuc_basecount']
merged_df = merged_df[final_cols]

#### Remove entries with missing harmonized entries

In [8]:
merged_cleaned_df = merged_df[merged_df['scientificName'] != '-']

### Process OTLs

##### List OTL's Phylums

In [9]:
harmonized_folder_path = "/home/camilababo/Documents/DNAquaIMG/countries-otls/harmonized"

In [10]:
def get_unique_phylum_values(folder_path: str) -> set:
    """
    Recursively reads all .tsv files in a folder, and subfolders, and extracts unique values from
    the 'phylum' column, and returns a set of unique values.

    Parameters:
        folder_path (str): Path to the folder containing .tsv files.

    Returns:
        set: A set of unique values from the 'phylum' column across all files.
    """
    unique_phyla = set()

    for root, _, files in os.walk(folder_path):
        for file_name in files:
            if file_name.endswith(".tsv"):
                file_path = os.path.join(root, file_name)

                try:
                    df = pd.read_csv(file_path, sep='\t')

                    if 'phylum' in df.columns:
                        # Add unique values from the 'phylum' column to the set
                        unique_phyla.update(df['phylum'].dropna().unique())
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

    return unique_phyla

In [11]:
unique_phyla = get_unique_phylum_values(harmonized_folder_path)
unique_phyla.remove('-')

In [12]:
unique_phyla

{'Annelida',
 'Arthropoda',
 'Bryozoa',
 'Charophyta',
 'Chlorophyta',
 'Chordata',
 'Cnidaria',
 'Cyanobacteria',
 'Entoprocta',
 'Mollusca',
 'Nematoda',
 'Nematomorpha',
 'Nemertea',
 'Ochrophyta',
 'Platyhelminthes',
 'Porifera',
 'Rhodophyta',
 'Tardigrada',
 'Tracheophyta'}

In [13]:
# Remove 'chordata'. Only phylum associated with inertebrates (BMI) and unicellular organisms (DIA) are to be considered.
unique_phyla.remove('Chordata')

#### Exploratory analysis on entries where no harmonized entry was retrieved ('-')

In [22]:
def calculate_missing_scientific_names(folder_path):
    results = []

    # Walk through all subdirectories and files in the root folder
    for dirpath, _, filenames in os.walk(folder_path):
        for file_name in filenames:
            if file_name.endswith(".tsv"):
                file_path = os.path.join(dirpath, file_name)

                try:
                    df = pd.read_csv(file_path, sep='\t')

                    if 'scientificName' in df.columns:
                        total_entries = len(df)
                        missing_entries = (df['scientificName'] == '-').sum()

                        percentage_missing = (missing_entries / total_entries) * 100 if total_entries > 0 else 0

                        file_name = os.path.basename(file_path)

                        results.append({
                            'File Name': file_name,
                            'Percentage Missing': round(percentage_missing, 1)
                        })
                    else:
                        results.append({
                            'File Name': file_name,
                            'Percentage Missing': 'Column not found'
                        })
                except Exception as e:
                    results.append({
                        'File Name': file_name,
                        'Percentage Missing': f'Error: {e}'
                    })

    return pd.DataFrame(results)

In [25]:
stats_on_missing_entries = calculate_missing_scientific_names(harmonized_folder_path)

In [29]:
stats_on_missing_entries.sort_values(by="Percentage Missing", ascending=False)

,File Name,Percentage Missing
7,France_taxonlists_bmi_harmonized.tsv,8.7
1,Austria_taxonlist_bmi_harmonized.tsv,8.6
2,Poland_taxonlists_bmi_harmonized.tsv,4.9
0,Sweden_taxonlist_bmi_harmonized.tsv,3.8
4,Germany_taxonlist_bmi_harmonized.tsv,3.8
10,Czech_taxonlist_diatoms_harmonized.tsv,3.8
5,Portugal_taxonlist_bmi_harmonized.tsv,2.9
6,Finland_taxonlist_bmi_harmonized.tsv,2.7
9,Ireland_taxonlist_diatoms_harmonized.tsv,2.6
11,Finland_taxonlist_diatoms_harmonized.tsv,2.5


#### Remove entries where no harmonized entry was retrieved ('-')

In [30]:
def remove_missing_scientific_names(folder_path):
    # Walk through all subdirectories and files in the root folder
    for dirpath, _, filenames in os.walk(folder_path):
        for file_name in filenames:
            if file_name.endswith(".tsv"):
                file_path = os.path.join(dirpath, file_name)

                try:
                    df = pd.read_csv(file_path, sep='\t')

                    if 'scientificName' in df.columns:
                        cleaned_df = df[df['scientificName'] != '-']

                        cleaned_df.to_csv(file_path, sep='\t', index=False)
                except Exception as e:
                    print(f"Error processing file {file_name}: {e}")

In [31]:
remove_missing_scientific_names(harmonized_folder_path)

#### Filter BOLD by OTL phylum set

In [14]:
def filter_dataframe_by_set_values(df, phylum_set):
    """
    Filters the BOLD DataFrame to keep only rows where the values in the 'phylum' column
    are in the provided set of allowed phylum values from the OTL.

    Parameters:
    - df (pd.DataFrame): The DataFrame to filter.
    - phylum_set (set): The set of allowed values for the phylum.

    Returns:
    - pd.DataFrame: The filtered DataFrame.
    """
    if 'phylum' in df.columns:
        return df[df['phylum'].isin(phylum_set)]
    else:
        raise ValueError(f"Column 'phylum' not found in the DataFrame.")

In [15]:
merged_cleaned_phyl_filtered_df = filter_dataframe_by_set_values(merged_cleaned_df, unique_phyla)

In [ ]:
# Save file
chunk_size = 100000
with open("/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/bold-final.tsv", "w") as f:
    merged_cleaned_phyl_filtered_df.head(0).to_csv(f, sep="\t", index=False)
    for i in range(0, len(merged_cleaned_phyl_filtered_df), chunk_size):
        merged_cleaned_phyl_filtered_df.iloc[i:i+chunk_size].to_csv(f, sep="\t", index=False, header=False)

#### Check values with unique entries for 'identification method'

In [20]:
unique_idm_counts = merged_cleaned_phyl_filtered_df['identification_method'].value_counts()

unique_idm_counts.to_csv(
    "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/external_scripts/final-unique-idm.tsv",
    sep='\t',
    header=['Count'],
    index_label='Identification Method'
)

In [21]:
count_with_gap = merged_cleaned_phyl_filtered_df['nuc'].str.contains('-').sum()

count_with_gap

np.int64(2918763)

In [22]:
merged_cleaned_phyl_filtered_df.shape

(12718771, 14)

In [24]:
(2918763/12718771) * 100

22.948467269361167